In [ ]:
import os
os.makedirs('/tmp/l3', exist_ok=True)
open('/tmp/l3/build_occ_dataset.py','w',encoding='utf-8').write('"""Stage F data: BEV occupancy inputs and labels for all 404 nuScenes-mini keyframes (ego frame, 200x200 cells of 0.5 m, rows +x, cols +y, +/-50 m).\n\nINPUT (single keyframe only, no sweeps):\n  LiDAR channels : [log1p(#points with 0.3<z<3.0)/3, (max z +1)/5 where obstacle-height points exist, log1p(#all points)/3, mean intensity/255]\n  Radar channels : 5 radars, devkit filter (invalid_state==0, ambig_state==3): [log1p(#hits)/2, mean rcs/50, mean |v_comp|/20]\nLABEL (never seen by the model as input):\n  2 object : rasterised footprints of ALL annotated boxes of the 10 evaluated categories (real human GT)\n  1 static : cell hit by obstacle-height points (0.3<z<3.0 m, ego z=0 is ground) in >= 4 of the 11 ego-motion-compensated sweeps\n             (keyframe + up to 10 previous sweeps), and not on/next to an object footprint  (PSEUDO-GT, derived from LiDAR)\n  0 free   : everything else (includes cells never observed: occluded / out of range - the label cannot tell them apart)\nAll thresholds are fixed here, before any model is trained."""\nimport glob, json, os, sys\nimport numpy as np\nfrom scipy.ndimage import binary_dilation\n\nOUT = sys.argv[1]\nos.makedirs(OUT, exist_ok=True)\nV = None\nfor r, d, f in os.walk(\'/kaggle/input\'):\n    if \'v1.0-mini\' in d:\n        V = r; break\nROOT = V; TAB = os.path.join(ROOT, \'v1.0-mini\')\ntab = lambda n: json.load(open(os.path.join(TAB, n + \'.json\')))\nsample = {s[\'token\']: s for s in tab(\'sample\')}; scene = {s[\'token\']: s for s in tab(\'scene\')}\nsd = {d[\'token\']: d for d in tab(\'sample_data\')}; cs = {c[\'token\']: c for c in tab(\'calibrated_sensor\')}; ego = {e[\'token\']: e for e in tab(\'ego_pose\')}\ninst = {i[\'token\']: i for i in tab(\'instance\')}; cat = {c[\'token\']: c[\'name\'] for c in tab(\'category\')}\nann = {}\nfor a in tab(\'sample_annotation\'):\n    ann.setdefault(a[\'sample_token\'], []).append(a)\nfrom nuscenes.utils import splits\nGEN = {\'vehicle.car\', \'vehicle.truck\', \'vehicle.bus.bendy\', \'vehicle.bus.rigid\', \'vehicle.trailer\', \'vehicle.construction\', \'vehicle.motorcycle\', \'vehicle.bicycle\',\n       \'human.pedestrian.adult\', \'human.pedestrian.child\', \'human.pedestrian.construction_worker\', \'human.pedestrian.police_officer\', \'movable_object.trafficcone\', \'movable_object.barrier\'}\nNC, RES, EXT = 200, 0.5, 50.0\nBODY = np.array([[-1.0, 3.7], [-1.1, 1.1]])\nTY = {(\'F\', 4): \'<f4\', (\'F\', 8): \'<f8\', (\'I\', 1): \'i1\', (\'I\', 2): \'<i2\', (\'I\', 4): \'<i4\', (\'U\', 1): \'u1\', (\'U\', 2): \'<u2\', (\'U\', 4): \'<u4\'}\n\n\ndef qR(q):\n    w, x, y, z = q\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])\n\n\ndef read_pcd(p):\n    with open(p, \'rb\') as f:\n        h = {}\n        while True:\n            k, *v = f.readline().decode().strip().split(); h[k] = v\n            if k == \'DATA\':\n                break\n        dt = np.dtype([(n, TY[(t, int(s))]) for n, t, s in zip(h[\'FIELDS\'], h[\'TYPE\'], h[\'SIZE\'])])\n        return np.frombuffer(f.read(), dtype=dt, count=int(h[\'POINTS\'][0]))\n\n\ndef cells(xy):\n    r = np.floor((xy[:, 0] + EXT) / RES).astype(int); c = np.floor((xy[:, 1] + EXT) / RES).astype(int)\n    ok = (r >= 0) & (r < NC) & (c >= 0) & (c < NC)\n    return r[ok], c[ok], ok\n\n\ndef lidar_to_ego(sdr):\n    c = cs[sdr[\'calibrated_sensor_token\']]\n    raw = np.fromfile(os.path.join(ROOT, sdr[\'filename\']), dtype=np.float32).reshape(-1, 5)\n    return (qR(c[\'rotation\']) @ raw[:, :3].T).T + np.array(c[\'translation\']), raw[:, 3]\n\n\ndef strip_body(P, *extra):\n    m = ~((P[:, 0] > BODY[0, 0]) & (P[:, 0] < BODY[0, 1]) & (P[:, 1] > BODY[1, 0]) & (P[:, 1] < BODY[1, 1]))\n    return (P[m],) + tuple(e[m] for e in extra)\n\n\nxs = -EXT + (np.arange(NC) + 0.5) * RES; XX, YY = np.meshgrid(xs, xs, indexing=\'ij\')\n\n\ndef raster_boxes(boxes):\n    m = np.zeros((NC, NC), bool)\n    for cx, cy, w, l, yaw in boxes:\n        c, s = np.cos(yaw), np.sin(yaw); r0 = max(0, int((cx - l - w + EXT) / RES) - 2); r1 = min(NC, int((cx + l + w + EXT) / RES) + 3)\n        c0 = max(0, int((cy - l - w + EXT) / RES) - 2); c1 = min(NC, int((cy + l + w + EXT) / RES) + 3)\n        if r0 >= r1 or c0 >= c1:\n            continue\n        dx, dy = XX[r0:r1, c0:c1] - cx, YY[r0:r1, c0:c1] - cy\n        u = c * dx + s * dy; v = -s * dx + c * dy\n        m[r0:r1, c0:c1] |= (np.abs(u) <= l / 2) & (np.abs(v) <= w / 2)\n    return m\n\n\nmini_train, mini_val = set(splits.mini_train), set(splits.mini_val)\nitems = []\nfor t, s in sample.items():\n    nm = scene[s[\'scene_token\']][\'name\']\n    if nm in mini_train or nm in mini_val:\n        items.append((s[\'scene_token\'], s[\'timestamp\'], t, \'train\' if nm in mini_train else \'val\', nm))\nitems.sort()\nkey_lidar = {d[\'sample_token\']: d for d in sd.values() if d[\'is_key_frame\'] and \'/LIDAR_TOP/\' in d[\'filename\']}\nkey_radar = {}\nfor d in sd.values():\n    if d[\'is_key_frame\'] and \'/RADAR_\' in d[\'filename\']:\n        key_radar.setdefault(d[\'sample_token\'], []).append(d)\n\nN = len(items); XL = np.zeros((N, 4, NC, NC), np.float16); XR = np.zeros((N, 3, NC, NC), np.float16); Y = np.zeros((N, NC, NC), np.uint8)\nSTATS = []\nfor k, (_, ts, t, split, nm) in enumerate(items):\n    kl = key_lidar[t]; ek = ego[kl[\'ego_pose_token\']]; Rk, tk = qR(ek[\'rotation\']), np.array(ek[\'translation\'])\n    P, inten = lidar_to_ego(kl); P, inten = strip_body(P, inten)\n    obst = (P[:, 2] > 0.3) & (P[:, 2] < 3.0)\n    r, c, ok = cells(P[obst, :2]); cnt = np.zeros((NC, NC)); np.add.at(cnt, (r, c), 1)\n    mz = np.full((NC, NC), -np.inf); np.maximum.at(mz, (r, c), P[obst, 2][ok])\n    ra, ca, oka = cells(P[:, :2]); call = np.zeros((NC, NC)); np.add.at(call, (ra, ca), 1); isum = np.zeros((NC, NC)); np.add.at(isum, (ra, ca), inten[oka])\n    XL[k, 0] = np.log1p(cnt) / 3; XL[k, 1] = np.where(np.isfinite(mz), (mz + 1) / 5, 0); XL[k, 2] = np.log1p(call) / 3; XL[k, 3] = np.where(call > 0, isum / np.maximum(call, 1) / 255, 0)\n    # radar\n    hits = np.zeros((NC, NC)); rcs = np.zeros((NC, NC)); spd = np.zeros((NC, NC))\n    for rd in key_radar[t]:\n        a = read_pcd(os.path.join(ROOT, rd[\'filename\'])); a = a[(a[\'invalid_state\'] == 0) & (a[\'ambig_state\'] == 3)]\n        if len(a) == 0:\n            continue\n        cr = cs[rd[\'calibrated_sensor_token\']]; Pe = (qR(cr[\'rotation\']) @ np.stack([a[\'x\'], a[\'y\'], a[\'z\']]).astype(float)).T + np.array(cr[\'translation\'])\n        rr, cc, okr = cells(Pe[:, :2]); np.add.at(hits, (rr, cc), 1); np.add.at(rcs, (rr, cc), a[\'rcs\'][okr]); np.add.at(spd, (rr, cc), np.hypot(a[\'vx_comp\'], a[\'vy_comp\'])[okr])\n    XR[k, 0] = np.log1p(hits) / 2; XR[k, 1] = np.where(hits > 0, rcs / np.maximum(hits, 1) / 50, 0); XR[k, 2] = np.where(hits > 0, spd / np.maximum(hits, 1) / 20, 0)\n    # object labels (human GT)\n    boxes = []\n    for a in ann.get(t, []):\n        if cat[inst[a[\'instance_token\']][\'category_token\']] not in GEN:\n            continue\n        cen = Rk.T @ (np.array(a[\'translation\']) - tk); Rb = Rk.T @ qR(a[\'rotation\'])\n        boxes.append((cen[0], cen[1], a[\'size\'][0], a[\'size\'][1], float(np.arctan2(Rb[1, 0], Rb[0, 0]))))\n    obj = raster_boxes(boxes)\n    # static pseudo-GT: persistence over 11 ego-motion-compensated sweeps\n    persist = (cnt > 0).astype(int); token = kl[\'prev\']; nsw = 0\n    while token and nsw < 10:\n        sw = sd[token]; ep = ego[sw[\'ego_pose_token\']]\n        Pw, _ = lidar_to_ego(sw); Pg = (qR(ep[\'rotation\']) @ Pw.T).T + np.array(ep[\'translation\']); Pk = (Rk.T @ (Pg - tk).T).T\n        Pk, = strip_body(Pk); m = (Pk[:, 2] > 0.3) & (Pk[:, 2] < 3.0); rr, cc, _ = cells(Pk[m, :2]); h = np.zeros((NC, NC), bool); h[rr, cc] = True\n        persist += h; token = sw[\'prev\']; nsw += 1\n    static = (persist >= 4) & ~binary_dilation(obj, np.ones((3, 3), bool))\n    Y[k][static] = 1; Y[k][obj] = 2\n    STATS.append(dict(idx=k, token=t, split=split, scene=nm, sweeps=nsw, n_obj=len(boxes), obj_cells=int(obj.sum()), static_cells=int(static.sum())))\n    if k % 50 == 0:\n        print(\'built\', k, \'/\', N, flush=True)\nsplit = np.array([s[\'split\'] for s in STATS]); tokens = np.array([s[\'token\'] for s in STATS]); scenes = np.array([s[\'scene\'] for s in STATS])\nnp.savez_compressed(os.path.join(OUT, \'occ_dataset.npz\'), XL=XL, XR=XR, Y=Y, split=split, token=tokens, scene=scenes)\njson.dump(STATS, open(os.path.join(OUT, \'occ_dataset_stats.json\'), \'w\'))\ntr, va = split == \'train\', split == \'val\'\nprint(\'train\', tr.sum(), \'val\', va.sum(), \'| label frequency train free/static/object:\', [round(float((Y[tr] == c).mean()), 4) for c in range(3)], \'val:\', [round(float((Y[va] == c).mean()), 4) for c in range(3)])\nprint(\'mean sweeps used\', np.mean([s[\'sweeps\'] for s in STATS]), \'| DATASET BUILD PASS\')\n')
open('/tmp/l3/train_occ.py','w',encoding='utf-8').write('"""Stage F: train a small U-Net BEV occupancy network (free / static / object) from LiDAR (1 sweep) and/or radar BEV channels.\nFixed a priori: 80 epochs, batch 8, AdamW lr 2e-3 (cosine), weight decay 1e-4, seed 0, flip-y augmentation; last-epoch model is reported (no VAL-based selection)."""\nimport json, os, sys, time\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nDATA, OUT = sys.argv[1], sys.argv[2]\nos.makedirs(OUT, exist_ok=True)\ndev = \'cuda\'\nd = np.load(os.path.join(DATA, \'occ_dataset.npz\'))\nXL, XR, Y, split = d[\'XL\'].astype(np.float32), d[\'XR\'].astype(np.float32), d[\'Y\'].astype(np.int64), d[\'split\']\ntr, va = np.where(split == \'train\')[0], np.where(split == \'val\')[0]\nCH = {\'lidar_radar\': list(range(7)), \'lidar\': list(range(4)), \'radar\': [4, 5, 6]}\nX = np.concatenate([XL, XR], 1)\nEPOCHS, BS, LR = 80, 8, 2e-3\n\n\ndef block(i, o):\n    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))\n\n\nclass UNet(nn.Module):\n    def __init__(self, cin, base=32, ncls=3):\n        super().__init__()\n        c = [base, base * 2, base * 4, base * 8]\n        self.e1, self.e2, self.e3, self.e4 = block(cin, c[0]), block(c[0], c[1]), block(c[1], c[2]), block(c[2], c[3])\n        self.u3, self.d3 = nn.ConvTranspose2d(c[3], c[2], 2, 2), block(c[3], c[2])\n        self.u2, self.d2 = nn.ConvTranspose2d(c[2], c[1], 2, 2), block(c[2], c[1])\n        self.u1, self.d1 = nn.ConvTranspose2d(c[1], c[0], 2, 2), block(c[1], c[0])\n        self.out = nn.Conv2d(c[0], ncls, 1)\n\n    def forward(self, x):\n        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2)); e4 = self.e4(F.max_pool2d(e3, 2))\n        x = self.d3(torch.cat([self.u3(e4), e3], 1)); x = self.d2(torch.cat([self.u2(x), e2], 1)); x = self.d1(torch.cat([self.u1(x), e1], 1))\n        return self.out(x)\n\n\ndef metrics(pred, gt):\n    """pred, gt: (n,200,200) int. strict per-cell IoU + 1-cell-tolerant binary occupied precision/recall."""\n    out = {}\n    ious = []\n    for c, nm in enumerate((\'free\', \'static\', \'object\')):\n        i = ((pred == c) & (gt == c)).sum(); u = ((pred == c) | (gt == c)).sum(); ious.append(i / max(1, u)); out[f\'IoU_{nm}\'] = round(float(i / max(1, u)), 4)\n    out[\'mIoU\'] = round(float(np.mean(ious)), 4); out[\'mIoU_occupied_classes\'] = round(float(np.mean(ious[1:])), 4)\n    po, go = pred > 0, gt > 0; i = (po & go).sum(); u = (po | go).sum()\n    out[\'IoU_occupied(static+object)\'] = round(float(i / max(1, u)), 4)\n    out[\'occupied_precision\'] = round(float(i / max(1, po.sum())), 4); out[\'occupied_recall\'] = round(float(i / max(1, go.sum())), 4)\n    from scipy.ndimage import binary_dilation\n    k = np.ones((1, 3, 3), bool)\n    out[\'occupied_precision_tol1\'] = round(float((po & binary_dilation(go, k)).sum() / max(1, po.sum())), 4); out[\'occupied_recall_tol1\'] = round(float((go & binary_dilation(po, k)).sum() / max(1, go.sum())), 4)\n    o_p, o_g = pred == 2, gt == 2\n    out[\'object_precision\'] = round(float((o_p & o_g).sum() / max(1, o_p.sum())), 4); out[\'object_recall\'] = round(float((o_p & o_g).sum() / max(1, o_g.sum())), 4)\n    return out\n\n\n@torch.no_grad()\ndef predict(model, x):\n    model.eval(); out = []\n    for i in range(0, len(x), 8):\n        out.append(model(torch.from_numpy(x[i:i + 8]).to(dev)).argmax(1).cpu().numpy())\n    return np.concatenate(out).astype(np.uint8)\n\n\nfreq = np.array([(Y[tr] == c).mean() for c in range(3)]); w = torch.tensor((1 / np.sqrt(freq)) / (1 / np.sqrt(freq)).sum() * 3, dtype=torch.float32, device=dev)\nprint(\'class freq\', freq.round(4), \'weights\', w.cpu().numpy().round(2))\nresults, logs, preds = {}, {}, {}\nYva = Y[va]\n# ---- baselines on the same VAL grids -------------------------------------------------------------------------------------\nrule = (XL[va][:, 0] > 0).astype(np.int64)   # any obstacle-height LiDAR point in the keyframe -> occupied\nresults[\'baseline_rule_single_sweep_obstacle_height\'] = {k: v for k, v in metrics(rule, Yva).items() if k.startswith((\'IoU_occupied\', \'occupied\'))}\nprint(\'rule baseline\', results[\'baseline_rule_single_sweep_obstacle_height\'])\nfor name, ch in CH.items():\n    torch.manual_seed(0); np.random.seed(0)\n    model = UNet(len(ch)).to(dev); nparam = sum(p.numel() for p in model.parameters())\n    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4); steps = EPOCHS * int(np.ceil(len(tr) / BS)); sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)\n    Xt = X[:, ch]; log = []\n    t0 = time.time()\n    for ep in range(1, EPOCHS + 1):\n        model.train(); perm = np.random.permutation(tr); tot = 0\n        for i in range(0, len(perm), BS):\n            b = perm[i:i + BS]; x = torch.from_numpy(Xt[b]).to(dev); y = torch.from_numpy(Y[b]).to(dev)\n            if np.random.rand() < 0.5:\n                x = torch.flip(x, [3]); y = torch.flip(y, [2])\n            lg = model(x); pr = F.softmax(lg, 1); oh = F.one_hot(y, 3).permute(0, 3, 1, 2).float()\n            dice = 1 - ((2 * (pr * oh).sum((0, 2, 3)) + 1) / (pr.sum((0, 2, 3)) + oh.sum((0, 2, 3)) + 1))[1:].mean()\n            loss = F.cross_entropy(lg, y, weight=w) + dice\n            opt.zero_grad(); loss.backward(); opt.step(); sch.step(); tot += loss.item() * len(b)\n        rec = dict(epoch=ep, train_loss=tot / len(tr))\n        if ep % 10 == 0 or ep == 1:\n            m = metrics(predict(model, Xt[va]), Yva); rec.update(val_mIoU=m[\'mIoU\'], val_IoU_occupied=m[\'IoU_occupied(static+object)\'])\n        log.append(rec)\n        if ep % 20 == 0:\n            print(name, rec, flush=True)\n    secs = time.time() - t0\n    torch.save(dict(state_dict=model.state_dict(), channels=ch, variant=name, note=f\'Stage F {name}: U-Net BEV occupancy trained on nuScenes-mini TRAIN only, {EPOCHS} epochs\', classes=[\'free\', \'static\', \'object\']), os.path.join(OUT, f\'our_occupancy_{name}.pth\'))\n    # reload test: fresh model from the file must reproduce the predictions exactly\n    p1 = predict(model, Xt[va]); ck = torch.load(os.path.join(OUT, f\'our_occupancy_{name}.pth\'), map_location=dev)\n    m2 = UNet(len(ch)).to(dev); m2.load_state_dict(ck[\'state_dict\']); p2 = predict(m2, Xt[va]); reload_ok = bool((p1 == p2).all())\n    # latency\n    xb = torch.from_numpy(Xt[va[:1]]).to(dev); model.eval()\n    with torch.no_grad():\n        for _ in range(10): model(xb)\n        torch.cuda.synchronize(); t1 = time.time()\n        for _ in range(100): model(xb)\n        torch.cuda.synchronize(); lat = (time.time() - t1) / 100\n    results[name] = dict(metrics(p1, Yva), params_M=round(nparam / 1e6, 2), train_secs=round(secs), reload_identical=reload_ok, latency_ms_T4=round(lat * 1000, 2),\n                         final_train_loss=round(log[-1][\'train_loss\'], 4), peak_mem_MB=round(torch.cuda.max_memory_allocated() / 1e6))\n    logs[name] = log; preds[name] = p1\n    print(name, results[name], flush=True)\njson.dump(dict(results=results, logs=logs, class_freq_train=freq.tolist(), epochs=EPOCHS, batch=BS, lr=LR, n_train=int(len(tr)), n_val=int(len(va))), open(os.path.join(OUT, \'occupancy_metrics.json\'), \'w\'), indent=2)\nnp.savez_compressed(os.path.join(OUT, \'occ_val_predictions.npz\'), **{f\'pred_{k}\': v for k, v in preds.items()}, gt=Yva.astype(np.uint8), XL=XL[va].astype(np.float16), XR=XR[va].astype(np.float16), token=d[\'token\'][va], scene=d[\'scene\'][va])\nfig, ax = plt.subplots(1, 2, figsize=(13, 4.5))\nfor name, log in logs.items():\n    ax[0].plot([r[\'epoch\'] for r in log], [r[\'train_loss\'] for r in log], label=name)\n    e = [r for r in log if \'val_mIoU\' in r]; ax[1].plot([r[\'epoch\'] for r in e], [r[\'val_mIoU\'] for r in e], \'o-\', label=name)\nax[0].set_title(\'train loss\'); ax[0].set_yscale(\'log\'); ax[0].legend(); ax[1].set_title(\'mini VAL mIoU (monitoring only, no selection)\'); ax[1].legend(); ax[0].set_xlabel(\'epoch\'); ax[1].set_xlabel(\'epoch\')\nfig.tight_layout(); fig.savefig(os.path.join(OUT, \'occupancy_training_curves.png\'), dpi=100)\nprint(\'OCCUPANCY TRAIN PASS\')\n')
print('written')

In [ ]:
%%bash
set -euo pipefail
export W=/tmp/l3 MPLBACKEND=Agg CUDA_VISIBLE_DEVICES=0
mkdir -p /kaggle/working/outputs $W/data
TAR=$(find /kaggle/input -maxdepth 6 -name 'env_bevfusion.tar.*' | head -1); echo "$TAR"
case "$TAR" in *.zst) zstd -dc "$TAR" | tar -C $W -xf - envs/bevfusion ;; *.gz) tar -C $W -xzf "$TAR" envs/bevfusion ;; esac
PY=$W/envs/bevfusion/bin/python
$PY -c "import torch, scipy, numpy; print('torch', torch.__version__, torch.cuda.get_device_name(0), 'numpy', numpy.__version__)"
$PY -W ignore $W/build_occ_dataset.py $W/data 2>&1 | tail -15
cp $W/data/occ_dataset_stats.json /kaggle/working/outputs/
$PY -W ignore $W/train_occ.py $W/data /kaggle/working/outputs 2>&1 | tail -30
ls -la /kaggle/working/outputs
